## 목적 및 작업 원칙

- 입력: `data/combined_2024_2025.csv` (103,939행 × 82컬럼, 2024/2025년 국민여행조사 리사이징 통합 데이터)
- 이 노트북은 아래 **4가지 전처리 작업만** 수행합니다: 결측치 처리 / 이상치 처리 / 손상 레코드 처리 / 파생변수 생성
  (지역경제 상관관계 분석 등 후속 분석 단계에서 할 작업 — 예: log 변환, 4분면 분류, 회귀용 그룹핑 등 — 은 포함하지 않습니다.)
- 규칙
  1. 원본 컬럼명은 바꾸지 않는다
  2. 쓰지 않을 컬럼만 제외하고 나머지는 전부 남긴다
  3. 파생변수 이름만 영문으로 짓는다
  4. 산출물은 `combined_2024_2025.csv` 하나로만 만든다
- **작업 범위**: 이번 전처리는 **1차 여행(`D_TRA1_*`) 관련 컬럼을 중심**으로 결측치·이상치·손상 레코드를 점검합니다. 2~6차 여행(`D_TRA2~6_*`) 컬럼은 후속 분석(기술통계 등)을 위해 리사이징 단계에서 별도로 포함된 보조 데이터이므로, 원본 값을 그대로 보존하고 별도의 이상치/손상 레코드 점검은 하지 않습니다. (단, 사용자 요청에 따른 숫자형 dtype 정리는 전체 컬럼에 공통 적용합니다.)

## [1] 데이터 불러오기 및 기본 정보 확인

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

df_raw = pd.read_csv('data/combined_2024_2025.csv')
print('shape:', df_raw.shape)
df_raw.info()

shape: (103939, 82)
<class 'pandas.DataFrame'>
RangeIndex: 103939 entries, 0 to 103938
Data columns (total 82 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ID                 103939 non-null  str    
 1   WT_DOM             103939 non-null  float64
 2   BSEX               103939 non-null  int64  
 3   BAGE               103939 non-null  int64  
 4   BINC1              103939 non-null  int64  
 5   D_TRA1_SYEAR       53055 non-null   float64
 6   D_TRA1_SMONTH      53055 non-null   float64
 7   D_TRA1_1_SPOT      53055 non-null   float64
 8   D_TRA1_CASE        53055 non-null   float64
 9   D_TRA1_COST        53055 non-null   float64
 10  D_TRA1_NUM         53055 non-null   float64
 11  D_TRA1_ONE_COST    53055 non-null   float64
 12  D_TRA2_SYEAR       3014 non-null    float64
 13  D_TRA2_SMONTH      3014 non-null    float64
 14  D_TRA2_1_SPOT      3014 non-null    float64
 15  D_TRA2_CASE        3014 non-null    float6

### 컬럼 정의표

원본 82개 컬럼에 대한 설명입니다 (출처: `combined_2024_2025.csv 컬럼 요약 정보` 문서 + 코드북). 회차별 반복 컬럼(1~6차 각 7개, 지역별 반복 컬럼 각 17개)은 묶어서 표기합니다.

| 컬럼명 | 한글 설명 | 원본 타입 | 처리 후 타입 | 비고 |
|---|---|---|---|---|
| `ID` | 응답자 고유 식별 번호 | object | object | 그대로 유지 |
| `WT_DOM` | 국내여행 가중치 | float64 | float64 | **정수 변환 제외** (가중치는 연속값) |
| `BSEX` | 성별 (1:남성, 2:여성) | int64 | int64 | 이미 정수, 결측 0건 |
| `BAGE` | 연령대 (1~7 구간) | int64 | int64 | 이미 정수, 결측 0건 |
| `BINC1` | 가구 소득 구간 (1~7 구간) | int64 | int64 | 이미 정수, 결측 0건 |
| `D_TRA{n}_SYEAR` (n=1~6) | n차 여행 시작 연도 | float64 | Int64 | 결측 = 해당 회차 여행 없음 |
| `D_TRA{n}_SMONTH` (n=1~6) | n차 여행 시작 월(1~12) | float64 | Int64 | 〃 |
| `D_TRA{n}_1_SPOT` (n=1~6) | n차 여행 주요 방문지 코드(시도2자리+시군구3자리) | float64 | Int64 | 〃 |
| `D_TRA{n}_CASE` (n=1~6) | n차 여행 목적/유형 (1~5) | float64 | Int64 | 〃 |
| `D_TRA{n}_COST` (n=1~6) | n차 여행 총 지출 비용(원) | float64 | Int64 | 〃 |
| `D_TRA{n}_NUM` (n=1~6) | n차 여행 동반 포함 총 인원수 | float64 | Int64 | 〃 |
| `D_TRA{n}_ONE_COST` (n=1~6) | n차 여행 1인당 평균 지출 비용(원) = COST/NUM | float64 | Int64 | 〃 |
| `국내_A_여행횟수_관광전체_{시도}` (17개) | 해당 시도를 1차 방문지로 하는 관광 포함 여행 횟수 | int64 | int64 | 이미 정수, 결측 0건 |
| `국내_A_여행지출_관광전체_{시도}` (17개) | 해당 시도를 1차 방문지로 하는 관광 포함 여행의 1인당 지출 합계(원) | float64 | float64 | **정수 변환 제외** (소수점 실측값 존재) |
| `연도` | 조사 대상 연도(2024/2025) | int64 | int64 | 이미 정수, 결측 0건 |

**사용여부**: 82개 컬럼 모두 결측/이상치 없이 유효하거나(앞서 실측 확인) 후속 분석(지역경제 상관관계 분석)에 필요할 수 있는 컬럼이라 판단되어, **제외하는 컬럼 없이 전부 사용**합니다.

## [2] EDA — 결측치 / 이상치 / 손상 레코드 확인 및 파생변수 설계

### 결측치

#### 결측치 확인

In [2]:
# 컬럼별 결측치 개수
na_counts = df_raw.isna().sum()
print('결측치가 있는 컬럼 수:', (na_counts > 0).sum(), '/', len(na_counts))
print()
print(na_counts[na_counts > 0])

결측치가 있는 컬럼 수: 42 / 82

D_TRA1_SYEAR        50884
D_TRA1_SMONTH       50884
D_TRA1_1_SPOT       50884
D_TRA1_CASE         50884
D_TRA1_COST         50884
D_TRA1_NUM          50884
D_TRA1_ONE_COST     50884
D_TRA2_SYEAR       100925
D_TRA2_SMONTH      100925
D_TRA2_1_SPOT      100925
D_TRA2_CASE        100925
D_TRA2_COST        100925
D_TRA2_NUM         100925
D_TRA2_ONE_COST    100925
D_TRA3_SYEAR       103729
D_TRA3_SMONTH      103729
D_TRA3_1_SPOT      103729
D_TRA3_CASE        103729
D_TRA3_COST        103729
D_TRA3_NUM         103729
D_TRA3_ONE_COST    103729
D_TRA4_SYEAR       103911
D_TRA4_SMONTH      103911
D_TRA4_1_SPOT      103911
D_TRA4_CASE        103911
D_TRA4_COST        103911
D_TRA4_NUM         103911
D_TRA4_ONE_COST    103911
D_TRA5_SYEAR       103932
D_TRA5_SMONTH      103932
D_TRA5_1_SPOT      103932
D_TRA5_CASE        103932
D_TRA5_COST        103932
D_TRA5_NUM         103932
D_TRA5_ONE_COST    103932
D_TRA6_SYEAR       103938
D_TRA6_SMONTH      103938
D_TRA6_1_SPOT  

##### `D_TRA1_SYEAR/SMONTH/1_SPOT/CASE/COST/NUM/ONE_COST` (1차 여행 관련 7개 컬럼): 50,884건 (48.96%)

이 7개 컬럼은 결측 개수가 정확히 동일합니다 — 즉 "일부 항목만 무응답"이 아니라, **1차 여행 자체를 하지 않은 응답자는 7개 컬럼이 전부 함께 결측**되는 구조입니다. 아래 코드로 실제 부분결측(7개 중 일부만 NaN인 행)이 0건임을 확인합니다.

1. 평균/중앙값 등으로 대체
    - "여행을 하지 않았다"는 사실 자체가 정보인데, 임의의 지출/인원 값을 채우면 실제로 존재하지 않는 여행을 만들어내는 셈이라 왜곡이 큼
2. 행 삭제
    - 전체의 절반에 가까운 응답자가 그냥 "1차 여행을 하지 않은" 사람들이라, 삭제하면 무여행자 특성 자체가 데이터에서 사라짐(표본이 여행자로만 편향됨)
3. **[v] 그대로 유지(NaN)**
    - NaN이 "무응답/결측"이 아니라 "해당 회차 여행 없음"이라는 유효한 정보이므로, 대체하거나 삭제하지 않고 그대로 둠. 이후 분석 단계에서 여행경험 유무를 이 NaN 여부로 바로 판별할 수 있음

In [3]:
# 1차 여행 7개 컬럼이 "전부 함께" 결측인지(부분결측이 없는지) 검증
cols_trip1 = ['D_TRA1_SYEAR', 'D_TRA1_SMONTH', 'D_TRA1_1_SPOT', 'D_TRA1_CASE',
              'D_TRA1_COST', 'D_TRA1_NUM', 'D_TRA1_ONE_COST']
na_per_row = df_raw[cols_trip1].isna().sum(axis=1)
print('1차 여행 컬럼 결측 패턴 (0=전부 있음, 7=전부 없음):')
print(na_per_row.value_counts().sort_index())
print('부분결측(1~6개만 결측)인 행:', ((na_per_row > 0) & (na_per_row < 7)).sum(), '건 -> 없으면 순수 구조적 결측')

# 참고: 2~6차 여행 컬럼도 동일한 구조인지만 가볍게 확인 (2~6차는 이번 전처리 대상 아님)
for n in range(2, 7):
    cols = [f'D_TRA{n}_SYEAR', f'D_TRA{n}_SMONTH', f'D_TRA{n}_1_SPOT', f'D_TRA{n}_CASE',
            f'D_TRA{n}_COST', f'D_TRA{n}_NUM', f'D_TRA{n}_ONE_COST']
    na_row = df_raw[cols].isna().sum(axis=1)
    partial = ((na_row > 0) & (na_row < 7)).sum()
    print(f'{n}차: 부분결측 {partial}건 (참고용, 이번 처리 대상 아님)')

1차 여행 컬럼 결측 패턴 (0=전부 있음, 7=전부 없음):
0    53055
7    50884
Name: count, dtype: int64
부분결측(1~6개만 결측)인 행: 0 건 -> 없으면 순수 구조적 결측
2차: 부분결측 0건 (참고용, 이번 처리 대상 아님)
3차: 부분결측 0건 (참고용, 이번 처리 대상 아님)
4차: 부분결측 0건 (참고용, 이번 처리 대상 아님)
5차: 부분결측 0건 (참고용, 이번 처리 대상 아님)
6차: 부분결측 0건 (참고용, 이번 처리 대상 아님)


#### 결측치 처리 기준

- 결측 처리 X (다른 82개 컬럼 전체): `ID`, `WT_DOM`, `BSEX`, `BAGE`, `BINC1`, `국내_A_여행횟수_관광전체_*`, `국내_A_여행지출_관광전체_*`, `연도` — 실측 결과 결측 0건
- 결측 처리 O: `D_TRA1_*` (7개, 48.96%) — **그대로 유지(NaN)**, 대체/삭제하지 않음. "1차 여행 없음"이라는 구조적 정보이기 때문
- `D_TRA2~6_*` (42개): 이번 전처리 대상 아님(원본 그대로 보존, dtype 변환만 공통 적용)

### 이상치

#### 이상치 확인

`D_TRA1_COST`(1차 여행 총 지출)와 `D_TRA1_NUM`(1차 여행 동반 인원)에 통계적 극단값이 존재하는지 IQR 기준으로 확인합니다.

In [4]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in ['D_TRA1_COST', 'D_TRA1_NUM', 'D_TRA1_ONE_COST']:
    s = df_raw[col].dropna()
    lo, hi = iqr_bounds(s)
    n_out = ((s < lo) | (s > hi)).sum()
    print(f'{col}: n={len(s)}, IQR 하한={lo:,.0f}, 상한={hi:,.0f}, 이상치={n_out}건 ({n_out/len(s):.1%}), max={s.max():,.0f}')

print()
print('D_TRA1_COST 상위 분포:')
print(df_raw['D_TRA1_COST'].dropna().describe(percentiles=[.9, .95, .99, .999]))
print()
print('D_TRA1_NUM 상위 분포:')
print(df_raw['D_TRA1_NUM'].dropna().describe(percentiles=[.9, .95, .99, .999]))

D_TRA1_COST: n=53055, IQR 하한=-237,500, 상한=662,500, 이상치=4981건 (9.4%), max=24,000,000
D_TRA1_NUM: n=53055, IQR 하한=-2, 상한=6, 이상치=523건 (1.0%), max=80


D_TRA1_ONE_COST: n=53055, IQR 하한=-100,000, 상한=300,000, 이상치=4067건 (7.7%), max=6,980,000

D_TRA1_COST 상위 분포:
count    5.305500e+04
mean     2.983116e+05
std      4.289676e+05
min      1.000000e+03
90%      6.323028e+05
95%      9.600000e+05
99%      2.000000e+06
99.9%    4.059460e+06
max      2.400000e+07
Name: D_TRA1_COST, dtype: float64

D_TRA1_NUM 상위 분포:
count    53055.000000
mean         2.432551
std          1.735315
min          1.000000
90%          4.000000
95%          4.000000
99%          6.000000
99.9%       20.000000
max         80.000000
Name: D_TRA1_NUM, dtype: float64


##### `D_TRA1_COST`(1차 여행비용) IQR 상한 초과: 4,981건(9.4%) / `D_TRA1_NUM`(동반인원) IQR 상한 초과: 523건(1.0%)

1. 상한 절단(winsorize/cap)
    - 원본 지출·인원수 값 자체가 바뀜. COST=NUM×ONE_COST 정합성이 100% 맞는 실측 응답인데, 임의로 잘라내면 실제로 응답자가 지출/신고한 값을 훼손하는 것
2. 이상치 행 삭제
    - COST 이상치가 9.4%나 되어 삭제 시 표본 손실이 크고, 단체여행·고가 여행은 국내여행조사에서 실재하는 현상이라 삭제할 근거가 약함 (리사이징 검토 리포트에서도 동일하게 삭제 비추천)
3. **[v] 원본 값 유지 + 이상치 플래그 파생변수만 추가**
    - COST/NUM 자체는 내적으로 완전히 일관되고(정합성 위반 0건) 유효범위(양수) 안에 있어 "오류"로 볼 근거가 없음. 다만 통계적으로 극단적인 값이라는 정보는 후속 분석(예: 평균 vs 중앙값 선택, robust 회귀 여부 판단)에 필요하므로 플래그만 남김

#### 이상치 처리 기준

- 이상치 처리 O: `D_TRA1_COST`, `D_TRA1_NUM` → 원본 값 유지 + IQR 기준 이상치 플래그 파생변수 추가(`TRIP1_COST_OUTLIER`, `TRIP1_NUM_OUTLIER`)
- 이상치 처리 X: `D_TRA1_ONE_COST`(COST/NUM 자동 계산값이라 COST 플래그로 대체 가능), `BSEX`/`BAGE`/`BINC1`/`D_TRA1_CASE`/`D_TRA1_1_SPOT`(사전 정의된 코드값이라 통계적 이상치 개념이 성립하지 않음, 유효범위 위반은 아래 손상 레코드에서 별도 확인)
- `D_TRA2~6_*`, `국내_A_*`: 이번 전처리 대상 아님

### 손상 레코드

#### 손상 레코드 확인

1차 여행 데이터에 대해 아래 5가지를 점검합니다.
1. **ID 중복** — 응답자 고유 식별번호가 중복되면 안 됨
2. **논리적 정합성** — `D_TRA1_COST = D_TRA1_NUM × D_TRA1_ONE_COST`가 성립해야 함
3. **물리적 유효범위** — `D_TRA1_NUM`(동반인원), `D_TRA1_COST`(총비용)는 0보다 커야 함
4. **코드 유효범위** — `D_TRA1_CASE`는 코드북상 1~5, `D_TRA1_1_SPOT`은 코드북 시/도 코드(11,21,22,23,24,25,26,29,31~39) 중 하나로 시작해야 함
5. **시점 정합성** — `D_TRA1_SYEAR`는 해당 행의 `연도`(조사 연도)와 같아야 함 (해당 연도 조사에서 다른 연도 여행이 대표여행으로 잡히면 이상)

In [5]:
VALID_SIDO_CODES = {11, 21, 22, 23, 24, 25, 26, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39}

# 1. ID 중복
print('1) ID 중복 건수:', df_raw['ID'].duplicated().sum())

# 2. COST = NUM * ONE_COST 정합성
m = df_raw['D_TRA1_COST'].notna()
calc_diff = (df_raw.loc[m, 'D_TRA1_COST'] - df_raw.loc[m, 'D_TRA1_NUM'] * df_raw.loc[m, 'D_TRA1_ONE_COST']).abs()
print('2) COST = NUM * ONE_COST 불일치(1원 초과):', (calc_diff > 1).sum(), '/', m.sum())

# 3. 물리적 유효범위(NUM, COST > 0)
print('3) NUM<=0 또는 COST<=0:', ((df_raw.loc[m, 'D_TRA1_NUM'] <= 0) | (df_raw.loc[m, 'D_TRA1_COST'] <= 0)).sum())

# 4. 코드 유효범위
case_bad = df_raw['D_TRA1_CASE'].notna() & ~df_raw['D_TRA1_CASE'].isin([1, 2, 3, 4, 5])
spot_sido = (df_raw['D_TRA1_1_SPOT'].dropna() // 1000).astype(int)
spot_bad = (~spot_sido.isin(VALID_SIDO_CODES)).sum()
print('4) CASE 유효범위(1~5) 위반:', case_bad.sum(), ' / SPOT 시도코드 유효범위 위반:', spot_bad)

# 5. 시점 정합성
syear_bad = df_raw['D_TRA1_SYEAR'].notna() & (df_raw['D_TRA1_SYEAR'] != df_raw['연도'])
print('5) D_TRA1_SYEAR != 연도:', syear_bad.sum())

1) ID 중복 건수: 0
2) COST = NUM * ONE_COST 불일치(1원 초과): 0 / 53055
3) NUM<=0 또는 COST<=0: 0
4) CASE 유효범위(1~5) 위반: 0  / SPOT 시도코드 유효범위 위반: 0
5) D_TRA1_SYEAR != 연도: 0


#### 손상 레코드 처리 기준

5가지 점검 모두 위반 0건으로 확인되어, **1차 여행 데이터에서는 삭제하거나 수정해야 할 손상 레코드가 없습니다.** 별도 처리 없이 원본을 그대로 사용합니다.

(참고: `D_TRA2~6_*`나 지역별 사전집계 컬럼(`국내_A_*`)을 함께 볼 때는 회차 순서가 어긋나는 소수 사례나 CASE 기준 여행 수와 지역 합계 여행 수가 정확히 일치하지 않는 소수 사례가 있으나, 이번 전처리는 1차 여행 중심으로 범위를 좁혔으므로 다루지 않습니다.)

### 파생변수 설계

| 파생변수명 | 의미 | 생성 로직 | 값이 없는 경우 |
|---|---|---|---|
| `TRIP1_SIDO_CD` | 1차 여행 방문지의 시/도 코드(2자리) | `D_TRA1_1_SPOT // 1000` | `D_TRA1_1_SPOT`이 결측이면 결측 |
| `TRIP1_SIDO_NM` | 1차 여행 방문지의 시/도명(한글) | 코드북 시/도 코드표로 `TRIP1_SIDO_CD` 매핑 | 〃 |
| `TRIP1_SIDO_MONTH` | **지역×월 교차 변수** — 1차 여행이 어느 지역에 몇 월에 있었는지 하나의 값으로 표현 | `TRIP1_SIDO_NM + "_" + D_TRA1_SMONTH월"` (예: `강원_1월`). 기존 `D_TRA1_SMONTH`(월)와 새로 만든 `TRIP1_SIDO_NM`(지역)을 그대로 결합한 것이라 별도의 "월" 파생변수는 만들지 않음 | `D_TRA1_1_SPOT` 또는 `D_TRA1_SMONTH`가 결측이면 결측 |
| `TRIP1_COST_OUTLIER` | 1차 여행비용(`D_TRA1_COST`) IQR 이상치 여부 | `D_TRA1_COST`가 IQR 상한(662,500원) 초과면 1, 아니면 0. 1차 여행이 없으면 결측 | `D_TRA1_COST`가 결측이면 결측 |
| `TRIP1_NUM_OUTLIER` | 1차 여행 동반인원(`D_TRA1_NUM`) IQR 이상치 여부 | `D_TRA1_NUM`이 IQR 상한(6명) 초과면 1, 아니면 0 | `D_TRA1_NUM`이 결측이면 결측 |

`TRIP1_SIDO_CD`/`NM`은 `D_TRA1_1_SPOT` 코드의 앞 2자리가 코드북의 시/도 코드와 정확히 일치하는 것을 앞서 실측으로 확인한 뒤 만드는 파생변수입니다(코드북에 "SPOT 앞자리 = 시/도"라는 문구가 명시된 건 아니지만, 코드북의 시/도+시군구 코드 결합 규칙과 100% 일치하는 패턴이라 이 방식으로 생성합니다).

## [3] 전처리 적용 및 cleaned 데이터셋 생성

### 데이터 불러오기

In [6]:
df = pd.read_csv('data/combined_2024_2025.csv')
print('원본 shape:', df.shape)

원본 shape: (103939, 82)


### 숫자형 컬럼 dtype 정리

`WT_DOM`(가중치)과 `국내_A_여행지출_관광전체_*`(지역별 1인당 지출 합계 — 실제 소수점 값 존재)를 제외한 나머지 float 컬럼은 전부 정수값만 가지므로, 결측(NaN)은 그대로 유지하면서 정수로 표시되는 **nullable 정수형(`Int64`)** 으로 변환합니다. 단순히 `int64`로 바꾸면 결측행이 0 등으로 둔갑하므로 반드시 `Int64`를 사용합니다.

In [7]:
# float로 유지할 컬럼 (가중치 + 지역별 지출 합계: 실제 소수점 값 존재)
KEEP_FLOAT_COLS = ['WT_DOM'] + [c for c in df.columns if c.startswith('국내_A_여행지출_관광전체_')]

# D_TRA1~6의 SYEAR/SMONTH/1_SPOT/CASE/COST/NUM/ONE_COST -> Int64 (nullable) 변환 대상
TRIP_SUFFIXES = ['SYEAR', 'SMONTH', '1_SPOT', 'CASE', 'COST', 'NUM', 'ONE_COST']
int_convert_cols = [f'D_TRA{n}_{suf}' for n in range(1, 7) for suf in TRIP_SUFFIXES]

assert all(c in df.columns for c in int_convert_cols)
assert set(int_convert_cols).isdisjoint(KEEP_FLOAT_COLS)

before_dtypes = df.dtypes.copy()
for c in int_convert_cols:
    df[c] = df[c].astype('Int64')

print('Int64로 변환한 컬럼 수:', len(int_convert_cols))
print('float64로 유지한 컬럼 수:', len(KEEP_FLOAT_COLS))
print()
print(df[int_convert_cols[:7] + KEEP_FLOAT_COLS[:2]].dtypes)

Int64로 변환한 컬럼 수: 42
float64로 유지한 컬럼 수: 18

D_TRA1_SYEAR           Int64
D_TRA1_SMONTH          Int64
D_TRA1_1_SPOT          Int64
D_TRA1_CASE            Int64
D_TRA1_COST            Int64
D_TRA1_NUM             Int64
D_TRA1_ONE_COST        Int64
WT_DOM               float64
국내_A_여행지출_관광전체_서울    float64
dtype: object


### 결측치 처리

[2]에서 검토한 대로 `D_TRA1_*`의 NaN은 "1차 여행 없음"을 뜻하는 구조적 결측이므로 대체/삭제하지 않고 그대로 둡니다. 아래는 처리 없이 그대로임을 재확인하는 코드입니다.

In [8]:
# 처리 없음 - 원본 결측 개수와 동일한지만 재확인
assert df['D_TRA1_SYEAR'].isna().sum() == 50884
print('D_TRA1_* 결측치: 처리하지 않고 원본 그대로 유지 (50,884건, 변화 없음)')

D_TRA1_* 결측치: 처리하지 않고 원본 그대로 유지 (50,884건, 변화 없음)


### 손상 레코드 처리

[2]에서 5가지 점검(ID 중복, COST=NUM×ONE_COST 정합성, 물리적 유효범위, CASE/SPOT 코드 유효범위, SYEAR-연도 정합성) 모두 위반 0건을 확인했습니다. 삭제/수정할 행이 없으므로, 여기서는 dtype 변환 이후에도 동일하게 위반이 없는지만 재검증합니다.

In [9]:
n_before = len(df)

assert df['ID'].duplicated().sum() == 0

m = df['D_TRA1_COST'].notna()
diff = (df.loc[m, 'D_TRA1_COST'] - df.loc[m, 'D_TRA1_NUM'] * df.loc[m, 'D_TRA1_ONE_COST']).abs()
assert (diff > 1).sum() == 0

assert ((df.loc[m, 'D_TRA1_NUM'] <= 0) | (df.loc[m, 'D_TRA1_COST'] <= 0)).sum() == 0

assert (df['D_TRA1_CASE'].notna() & ~df['D_TRA1_CASE'].isin([1, 2, 3, 4, 5])).sum() == 0
spot_sido_check = (df['D_TRA1_1_SPOT'].dropna() // 1000).astype(int)
assert (~spot_sido_check.isin(VALID_SIDO_CODES)).sum() == 0

assert (df['D_TRA1_SYEAR'].notna() & (df['D_TRA1_SYEAR'] != df['연도'])).sum() == 0

# 손상 레코드로 삭제한 행 없음 -> 행 수 변화 없음
assert len(df) == n_before
print(f'손상 레코드 재검증 통과. 삭제된 행 없음 (행 수 {n_before:,} 유지)')

손상 레코드 재검증 통과. 삭제된 행 없음 (행 수 103,939 유지)


### 파생 변수 생성

`TRIP1_SIDO_CD` / `TRIP1_SIDO_NM` / `TRIP1_SIDO_MONTH` (지역×월 교차) / `TRIP1_COST_OUTLIER` / `TRIP1_NUM_OUTLIER`

In [10]:
# 코드북 "시/도 코드" 표 기준 매핑. 기존 국내_A_여행횟수_관광전체_{시도} / 국내_A_여행지출_관광전체_{시도}
# 컬럼들이 쓰는 시/도 약칭(서울, 부산, ...)과 동일한 표기로 맞춰서 다른 지역 컬럼과 바로 연결해 쓸 수 있게 함
SIDO_CD_TO_NM = {
    11: '서울', 21: '부산', 22: '대구', 23: '인천', 24: '광주', 25: '대전', 26: '울산', 29: '세종',
    31: '경기', 32: '강원', 33: '충북', 34: '충남', 35: '전북', 36: '전남', 37: '경북', 38: '경남', 39: '제주',
}
assert set(SIDO_CD_TO_NM) == VALID_SIDO_CODES

# TRIP1_SIDO_CD / TRIP1_SIDO_NM: D_TRA1_1_SPOT 앞 2자리(시/도 코드)를 분리하고 이름으로 매핑
spot_sido_cd = (df['D_TRA1_1_SPOT'] // 1000).astype('Int64')
df['TRIP1_SIDO_CD'] = spot_sido_cd
df['TRIP1_SIDO_NM'] = spot_sido_cd.map(SIDO_CD_TO_NM)

# TRIP1_SIDO_MONTH: 지역 x 월 교차 변수 (예: '강원_1월')
has_trip1 = df['TRIP1_SIDO_NM'].notna() & df['D_TRA1_SMONTH'].notna()
df['TRIP1_SIDO_MONTH'] = pd.NA
df.loc[has_trip1, 'TRIP1_SIDO_MONTH'] = (
    df.loc[has_trip1, 'TRIP1_SIDO_NM'] + '_' + df.loc[has_trip1, 'D_TRA1_SMONTH'].astype(str) + '월'
)

# TRIP1_COST_OUTLIER / TRIP1_NUM_OUTLIER: [2]에서 계산한 IQR 상한 기준 플래그(0/1)
cost_s = df['D_TRA1_COST'].dropna().astype(float)
num_s = df['D_TRA1_NUM'].dropna().astype(float)
_, cost_hi = iqr_bounds(cost_s)
_, num_hi = iqr_bounds(num_s)

df['TRIP1_COST_OUTLIER'] = pd.NA
df.loc[df['D_TRA1_COST'].notna(), 'TRIP1_COST_OUTLIER'] = (df['D_TRA1_COST'] > cost_hi).astype('Int64')
df['TRIP1_NUM_OUTLIER'] = pd.NA
df.loc[df['D_TRA1_NUM'].notna(), 'TRIP1_NUM_OUTLIER'] = (df['D_TRA1_NUM'] > num_hi).astype('Int64')
df['TRIP1_COST_OUTLIER'] = df['TRIP1_COST_OUTLIER'].astype('Int64')
df['TRIP1_NUM_OUTLIER'] = df['TRIP1_NUM_OUTLIER'].astype('Int64')

new_cols = ['TRIP1_SIDO_CD', 'TRIP1_SIDO_NM', 'TRIP1_SIDO_MONTH', 'TRIP1_COST_OUTLIER', 'TRIP1_NUM_OUTLIER']
print(df[new_cols].dtypes)
print()
print('결측 개수(모두 D_TRA1 결측 50,884건과 같아야 함):')
print(df[new_cols].isna().sum())
print()
print(df.loc[df['D_TRA1_SYEAR'].notna(), new_cols].head(8))

TRIP1_SIDO_CD          Int64
TRIP1_SIDO_NM            str
TRIP1_SIDO_MONTH      object
TRIP1_COST_OUTLIER     Int64
TRIP1_NUM_OUTLIER      Int64
dtype: object

결측 개수(모두 D_TRA1 결측 50,884건과 같아야 함):
TRIP1_SIDO_CD         50884
TRIP1_SIDO_NM         50884
TRIP1_SIDO_MONTH      50884
TRIP1_COST_OUTLIER    50884
TRIP1_NUM_OUTLIER     50884
dtype: int64

    TRIP1_SIDO_CD TRIP1_SIDO_NM TRIP1_SIDO_MONTH  TRIP1_COST_OUTLIER  TRIP1_NUM_OUTLIER
1              32            강원            강원_1월                   1                  0
2              32            강원            강원_1월                   1                  0
5              38            경남            경남_1월                   1                  0
11             31            경기            경기_2월                   0                  0
13             34            충남            충남_2월                   0                  0
16             36            전남            전남_2월                   0                  0
17             32            강원   

### 사용 컬럼 결정

[1]의 컬럼 정의표에서 정리한 대로, 원본 82개 컬럼 중 제외한 컬럼은 없습니다(전부 유효하거나 후속 분석에 필요할 수 있어 보존). 여기에 파생변수 5개를 추가합니다.

In [11]:
# 원본 82개 컬럼 모두 유지 확인 (이름/순서 변경 없음)
assert list(df.columns[:82]) == list(pd.read_csv('data/combined_2024_2025.csv', nrows=0).columns)

added_cols = [c for c in df.columns if c not in pd.read_csv('data/combined_2024_2025.csv', nrows=0).columns]
print('원본 컬럼 수:', 82)
print('추가된 파생변수:', added_cols)
print('최종 컬럼 수:', df.shape[1])

원본 컬럼 수: 82
추가된 파생변수: ['TRIP1_SIDO_CD', 'TRIP1_SIDO_NM', 'TRIP1_SIDO_MONTH', 'TRIP1_COST_OUTLIER', 'TRIP1_NUM_OUTLIER']
최종 컬럼 수: 87


### cleaned 데이터셋 저장

In [12]:
import os
os.makedirs('data/processed', exist_ok=True)
out_path = 'data/processed/combined_2024_2025_cleaned.csv'
df.to_csv(out_path, index=False)
print('저장 완료:', out_path)
print('shape:', df.shape)

저장 완료: data/processed/combined_2024_2025_cleaned.csv
shape: (103939, 87)


### 정리

| 단계 | 처리 내용 | 행 수 | 컬럼 수 |
|---|---|---|---|
| 원본 | - | 103,939 | 82 |
| dtype 정리 | float64 → Int64 변환 42개(`D_TRA1~6_*`), `WT_DOM`·`국내_A_여행지출_*` 18개는 float 유지 | 103,939 (±0) | 82 (±0) |
| 결측치 처리 | `D_TRA1_*` NaN(50,884건, 48.96%)은 "1차 여행 없음"을 뜻하는 구조적 결측이라 그대로 유지 | 103,939 (±0) | 82 (±0) |
| 이상치 처리 | `D_TRA1_COST`/`D_TRA1_NUM` 원본 값 유지, IQR 기준 이상치 플래그 파생변수 2개 추가 | 103,939 (±0) | +2 |
| 손상 레코드 처리 | ID 중복·COST=NUM×ONE_COST·유효범위·코드범위·시점정합성 5개 항목 모두 위반 0건 → 삭제 없음 | 103,939 (±0) | 82 (±0) |
| 파생 변수 생성 | `TRIP1_SIDO_CD`, `TRIP1_SIDO_NM`, `TRIP1_SIDO_MONTH`(지역×월 교차), `TRIP1_COST_OUTLIER`, `TRIP1_NUM_OUTLIER` | 103,939 (±0) | +5 |
| **최종** | `data/processed/combined_2024_2025_cleaned.csv` | **103,939** | **87** |

**요약**: 이번 전처리에서는 행을 하나도 삭제하지 않았습니다. 1차 여행 데이터는 결측(구조적 무여행)을 제외하면 정합성·유효범위·이상치 모두 실측으로 문제가 없다고 확인되어, 원본 값은 그대로 두고 정보를 명확히 하는 파생변수(dtype 정리, 이상치 플래그, 지역×월 교차)만 추가했습니다. `D_TRA2~6_*` 등 나머지 컬럼은 dtype만 정리했을 뿐 값은 원본 그대로입니다.